# Tool Result Reuse Cache: Practice Exercise

Build a tool result cache for a research assistant that uses **Tavily web search** and **Wikipedia** lookups. You will implement caching logic that stores and retrieves tool outputs based on the tool name and parameters.

**What you'll implement:**
- A cached tool wrapper that stores results with TTL (time-to-live)
- Cache key generation from tool name and parameters
- Cache invalidation based on TTL expiry

**Why this matters:**
- Tavily web searches cost ~$0.01 per query
- Wikipedia API calls add latency (500-1000ms)
- Caching tool results can reduce costs AND improve response times
- Users often ask similar questions - cache hits save real money

**Estimated time:** 15-20 minutes

## Setup

Run this cell to import all required libraries and initialize the Tavily and Wikipedia tools.

In [1]:
# Setup - run this cell first

import os
import time
import hashlib
import json
from datetime import datetime, timedelta
from typing import Any, Callable
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Load environment variables
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not found"

# In-memory cache storage (simulating Redis or similar)
# Structure: {cache_key: {"result": ..., "timestamp": ..., "ttl_seconds": ...}}
tool_cache: dict[str, dict] = {}

# Track statistics
cache_stats = {"hits": 0, "misses": 0, "expired": 0}

# Track actual API calls (to verify caching works)
api_call_counts = {"tavily": 0, "wikipedia": 0}

print("Setup complete!")
print("Cache storage initialized (in-memory dictionary)")
print("API call counters ready for tracking")

Setup complete!
Cache storage initialized (in-memory dictionary)
API call counters ready for tracking


## Initialize the Real Tools

We'll create the actual Tavily and Wikipedia tools, wrapped with call counting so we can verify caching works.

In [ ]:
# Initialize the actual tools
tavily_search = TavilySearch(max_results=3, topic="general")

wikipedia_api = WikipediaAPIWrapper(
    top_k_results=2,
    doc_content_chars_max=2000
)
wikipedia_tool = WikipediaQueryRun(api_wrapper=wikipedia_api)


# Wrapper functions that track API calls
def call_tavily(query: str) -> str:
    """Call Tavily search and track the API call."""
    api_call_counts["tavily"] += 1
    print(f"  [API CALL] Tavily search for: '{query[:50]}...'")
    
    response = tavily_search.invoke({"query": query})
    results = response.get("results", [])
    
    # Format results as string
    formatted = "\n\n".join([
        f"Title: {r.get('title', 'N/A')}\nURL: {r.get('url', 'N/A')}\nContent: {r.get('content', '')[:300]}..."
        for r in results
    ])
    return formatted if formatted else "No results found."


def call_wikipedia(query: str) -> str:
    """Call Wikipedia and track the API call."""
    api_call_counts["wikipedia"] += 1
    print(f"  [API CALL] Wikipedia lookup for: '{query[:50]}...'")
    
    return wikipedia_tool.invoke(query)


print("Tools initialized:")
print("  - call_tavily(query) - Web search via Tavily API")
print("  - call_wikipedia(query) - Encyclopedia lookup via Wikipedia API")

## Your Task: Implement the Tool Cache

You need to implement two functions:

### 1. `generate_cache_key(tool_name: str, **kwargs) -> str`
Generate a unique cache key from the tool name and its parameters.

**Requirements:**
- Combine tool name with sorted parameters to create a deterministic key
- Use MD5 or SHA256 hash for a compact key
- Example: `tavily:{"query": "AI news"}` -> `"tavily:a1b2c3d4..."`

### 2. `cached_tool_call(tool_name: str, tool_func: Callable, ttl_seconds: int, **kwargs) -> tuple[Any, bool]`
Execute a tool with caching. Check cache first, call tool on miss, store result.

**Requirements:**
- Generate cache key using `generate_cache_key`
- Check if key exists in `tool_cache` and is not expired
- On cache hit: return cached result and `True`
- On cache miss or expiry: call tool, store result with TTL, return result and `False`
- Update `cache_stats` appropriately

**TTL Guidelines from lecture:**
- Real-time data (news, current events): 15-30 minutes
- Semi-static data (Wikipedia facts): 6-24 hours
- The key insight: Wikipedia content rarely changes, but news is time-sensitive

In [ ]:
def generate_cache_key(tool_name: str, **kwargs) -> str:
    """
    Generate a unique cache key from tool name and parameters.
    
    Args:
        tool_name: Name of the tool being called (e.g., 'tavily', 'wikipedia')
        **kwargs: Tool parameters (e.g., query="AI news")
    
    Returns:
        A unique string key for caching
    
    Example:
        generate_cache_key("tavily", query="latest AI news")
        # Returns something like: "tavily:a1b2c3d4e5f6..."
    """
    # TODO: Implement cache key generation
    # 1. Sort kwargs to ensure deterministic ordering (use sorted(kwargs.items()))
    # 2. Convert sorted items to JSON string (use json.dumps())
    # 3. Create a hash (use hashlib.md5(string.encode()).hexdigest())
    # 4. Return f"{tool_name}:{hash_hex}"
    
    pass

In [ ]:
def cached_tool_call(
    tool_name: str,
    tool_func: Callable,
    ttl_seconds: int,
    **kwargs
) -> tuple[Any, bool]:
    """
    Execute a tool with caching support.
    
    Args:
        tool_name: Name of the tool (for cache key)
        tool_func: The actual tool function to call
        ttl_seconds: Time-to-live for cache entry in seconds
        **kwargs: Arguments to pass to the tool
    
    Returns:
        tuple: (result, is_cache_hit)
            - result: The tool's return value
            - is_cache_hit: True if result came from cache
    """
    # TODO: Implement cached tool execution
    # 
    # Step 1: Generate cache key
    # - Use generate_cache_key(tool_name, **kwargs)
    
    # Step 2: Check cache for existing entry
    # - Look up key in tool_cache dictionary
    # - If found, check if it's expired:
    #   - Get the timestamp and ttl_seconds from cached entry
    #   - Calculate expiry time: timestamp + timedelta(seconds=ttl_seconds)
    #   - Compare with current time: datetime.now()
    #   - If expired: increment cache_stats["expired"], treat as miss
    #   - If valid: increment cache_stats["hits"], return (cached_result, True)
    
    # Step 3: Cache miss - call the tool
    # - Increment cache_stats["misses"]
    # - Call tool_func(**kwargs) to get result
    
    # Step 4: Store result in cache
    # - Store in tool_cache[cache_key] with:
    #   - "result": the tool's return value
    #   - "timestamp": datetime.now()
    #   - "ttl_seconds": the TTL value
    
    # Step 5: Return (result, False) for cache miss
    
    pass

## Test Your Implementation

Run the tests below to verify your caching logic works correctly.

In [ ]:
# Test 1: Cache key generation
print("TEST 1: Cache Key Generation")
print("=" * 50)

key1 = generate_cache_key("tavily", query="AI news")
key2 = generate_cache_key("tavily", query="AI news")
key3 = generate_cache_key("tavily", query="different query")
key4 = generate_cache_key("wikipedia", query="AI news")

print(f"Key 1 (tavily, AI news): {key1}")
print(f"Key 2 (tavily, AI news): {key2}")
print(f"Key 3 (tavily, different): {key3}")
print(f"Key 4 (wikipedia, AI news): {key4}")

assert key1 == key2, "Same inputs should produce same key!"
assert key1 != key3, "Different queries should produce different keys!"
assert key1 != key4, "Different tools should produce different keys!"
print("\nAll cache key tests passed!")

In [ ]:
# Test 2: Wikipedia caching - hits and misses
print("\nTEST 2: Wikipedia Caching - Hits and Misses")
print("=" * 50)

# Reset counters
tool_cache.clear()
cache_stats.update({"hits": 0, "misses": 0, "expired": 0})
api_call_counts.update({"tavily": 0, "wikipedia": 0})

# First call - should be a miss (makes API call)
print("\nCall 1: First lookup for 'Alan Turing'")
result1, hit1 = cached_tool_call(
    "wikipedia",
    call_wikipedia,
    ttl_seconds=3600,  # 1 hour TTL for Wikipedia
    query="Alan Turing"
)
print(f"Cache hit: {hit1}")
print(f"Result preview: {result1[:100]}...")

# Second call with same query - should be a hit (no API call)
print("\nCall 2: Same lookup for 'Alan Turing'")
result2, hit2 = cached_tool_call(
    "wikipedia",
    call_wikipedia,
    ttl_seconds=3600,
    query="Alan Turing"
)
print(f"Cache hit: {hit2}")

# Third call with different query - should be a miss
print("\nCall 3: Different lookup for 'Marie Curie'")
result3, hit3 = cached_tool_call(
    "wikipedia",
    call_wikipedia,
    ttl_seconds=3600,
    query="Marie Curie"
)
print(f"Cache hit: {hit3}")

print(f"\nAPI calls made: {api_call_counts['wikipedia']}")
print(f"Cache stats: {cache_stats}")

assert not hit1, "First call should be a miss!"
assert hit2, "Second call with same query should be a hit!"
assert not hit3, "Call with different query should be a miss!"
assert api_call_counts["wikipedia"] == 2, "Should have made only 2 Wikipedia API calls!"
print("\nAll Wikipedia caching tests passed!")

In [ ]:
# Test 3: TTL expiration with short-lived cache
print("\nTEST 3: TTL Expiration")
print("=" * 50)

# Reset
tool_cache.clear()
cache_stats.update({"hits": 0, "misses": 0, "expired": 0})
api_call_counts.update({"tavily": 0, "wikipedia": 0})

# Call with very short TTL (simulating real-time data needs)
print("\nCall 1: Wikipedia lookup with 1-second TTL")
result1, hit1 = cached_tool_call(
    "wikipedia",
    call_wikipedia,
    ttl_seconds=1,  # Very short TTL for testing
    query="Python programming"
)
print(f"Cache hit: {hit1}")

# Immediate second call - should be a hit
print("\nCall 2: Immediate repeat (should hit cache)")
result2, hit2 = cached_tool_call(
    "wikipedia",
    call_wikipedia,
    ttl_seconds=1,
    query="Python programming"
)
print(f"Cache hit: {hit2}")

# Wait for TTL to expire
print("\nWaiting 2 seconds for TTL to expire...")
time.sleep(2)

# Third call after expiry - should be expired/miss
print("\nCall 3: After TTL expiry (should miss/expire)")
result3, hit3 = cached_tool_call(
    "wikipedia",
    call_wikipedia,
    ttl_seconds=1,
    query="Python programming"
)
print(f"Cache hit: {hit3}")

print(f"\nAPI calls made: {api_call_counts['wikipedia']}")
print(f"Cache stats: {cache_stats}")

assert not hit1, "First call should be a miss!"
assert hit2, "Immediate second call should be a hit!"
assert not hit3, "Call after TTL expiry should be a miss!"
assert cache_stats["expired"] >= 1, "Should have at least 1 expired entry!"
print("\nAll TTL expiration tests passed!")

## Integration: Cached Research Assistant

Now let's use your caching implementation in a research assistant scenario with both Tavily and Wikipedia.

In [ ]:
# Helper functions using your cached implementation

def cached_web_search(query: str) -> tuple[str, bool]:
    """
    Search the web using Tavily with caching.
    TTL: 30 minutes (news/current events change frequently)
    """
    return cached_tool_call(
        "tavily",
        call_tavily,
        ttl_seconds=1800,  # 30 minutes for current events
        query=query
    )


def cached_wiki_lookup(query: str) -> tuple[str, bool]:
    """
    Look up information on Wikipedia with caching.
    TTL: 6 hours (encyclopedic content is stable)
    """
    return cached_tool_call(
        "wikipedia",
        call_wikipedia,
        ttl_seconds=21600,  # 6 hours for encyclopedia content
        query=query
    )


print("Cached helper functions created:")
print("  - cached_web_search(query) - TTL: 30 minutes")
print("  - cached_wiki_lookup(query) - TTL: 6 hours")

In [ ]:
# Simulate a research session with repeated queries
print("\nSimulating Research Session")
print("=" * 60)

# Reset counters for clean test
tool_cache.clear()
cache_stats.update({"hits": 0, "misses": 0, "expired": 0})
api_call_counts.update({"tavily": 0, "wikipedia": 0})

# Simulate user queries in a research session
research_queries = [
    # First set of queries
    ("wikipedia", "Albert Einstein"),
    ("tavily", "latest AI breakthroughs 2024"),
    ("wikipedia", "Theory of Relativity"),
    
    # User asks similar questions again (should hit cache)
    ("wikipedia", "Albert Einstein"),  # Repeat - should be cached
    ("tavily", "latest AI breakthroughs 2024"),  # Repeat - should be cached
    
    # New queries
    ("wikipedia", "Quantum Mechanics"),
    ("tavily", "SpaceX news"),
    
    # More repeats
    ("wikipedia", "Theory of Relativity"),  # Repeat - should be cached
    ("wikipedia", "Quantum Mechanics"),  # Repeat - should be cached
]

print(f"Running {len(research_queries)} queries...\n")

for i, (tool, query) in enumerate(research_queries, 1):
    print(f"Query {i}: [{tool.upper()}] {query}")
    
    if tool == "wikipedia":
        result, is_hit = cached_wiki_lookup(query)
    else:
        result, is_hit = cached_web_search(query)
    
    status = "CACHE HIT" if is_hit else "CACHE MISS"
    print(f"  -> {status}\n")

print("=" * 60)
print("SESSION SUMMARY")
print("=" * 60)
print(f"Total queries: {len(research_queries)}")
print(f"Cache hits: {cache_stats['hits']}")
print(f"Cache misses: {cache_stats['misses']}")
print(f"Hit rate: {cache_stats['hits'] / len(research_queries) * 100:.1f}%")
print(f"\nActual API calls made:")
print(f"  Tavily: {api_call_counts['tavily']}")
print(f"  Wikipedia: {api_call_counts['wikipedia']}")
print(f"  Total: {sum(api_call_counts.values())}")
print(f"\nAPI calls saved: {len(research_queries) - sum(api_call_counts.values())}")

## Cost Savings Analysis

Let's calculate the real cost savings from caching Tavily and Wikipedia calls.

In [ ]:
# Cost analysis for research tools
print("TOOL RESULT CACHE - COST SAVINGS ANALYSIS")
print("=" * 60)

# Estimated costs per tool call
TOOL_COSTS = {
    "tavily": 0.01,     # ~$0.01 per Tavily search
    "wikipedia": 0.0,   # Wikipedia API is free, but has latency cost
}

LATENCY_MS = {
    "tavily": 800,      # ~800ms average latency
    "wikipedia": 500,   # ~500ms average latency
}

# Calculate from our session
total_queries = cache_stats["hits"] + cache_stats["misses"]
hit_rate = cache_stats["hits"] / total_queries if total_queries > 0 else 0

# Cost without caching
tavily_queries = sum(1 for t, _ in research_queries if t == "tavily")
cost_without_cache = tavily_queries * TOOL_COSTS["tavily"]

# Cost with caching
cost_with_cache = api_call_counts["tavily"] * TOOL_COSTS["tavily"]

# Latency savings
latency_saved_ms = (
    (tavily_queries - api_call_counts["tavily"]) * LATENCY_MS["tavily"] +
    (len([q for t, q in research_queries if t == "wikipedia"]) - api_call_counts["wikipedia"]) * LATENCY_MS["wikipedia"]
)

print(f"\nThis Session ({len(research_queries)} queries):")
print(f"  Cache hit rate: {hit_rate:.1%}")
print(f"  Tavily calls saved: {tavily_queries - api_call_counts['tavily']}")
print(f"  Wikipedia calls saved: {len([q for t, q in research_queries if t == 'wikipedia']) - api_call_counts['wikipedia']}")
print(f"\nCost Savings:")
print(f"  Without cache: ${cost_without_cache:.4f}")
print(f"  With cache: ${cost_with_cache:.4f}")
print(f"  Saved: ${cost_without_cache - cost_with_cache:.4f}")
print(f"\nLatency Savings:")
print(f"  Time saved: {latency_saved_ms}ms ({latency_saved_ms/1000:.1f}s)")

print(f"\n" + "=" * 60)
print("PROJECTED MONTHLY SAVINGS")
print("=" * 60)

# Project to monthly usage
daily_queries = 1000
monthly_queries = daily_queries * 30
projected_hit_rate = 0.40  # Conservative 40% hit rate

# Assume 60% Tavily, 40% Wikipedia split
monthly_tavily = monthly_queries * 0.6
monthly_wiki = monthly_queries * 0.4

monthly_without = monthly_tavily * TOOL_COSTS["tavily"]
monthly_with = monthly_tavily * (1 - projected_hit_rate) * TOOL_COSTS["tavily"]

print(f"\nAssumptions:")
print(f"  Daily queries: {daily_queries}")
print(f"  Projected hit rate: {projected_hit_rate:.0%}")
print(f"  Tool mix: 60% Tavily, 40% Wikipedia")
print(f"\nMonthly Projections:")
print(f"  Cost without cache: ${monthly_without:.2f}/month")
print(f"  Cost with cache: ${monthly_with:.2f}/month")
print(f"  Monthly savings: ${monthly_without - monthly_with:.2f}")
print(f"  Annual savings: ${(monthly_without - monthly_with) * 12:.2f}")

## Bonus: Integrating Cached Tools with an Agent

Once your caching implementation is working, here's how you can integrate it with a LangChain agent. The cached tools work transparently - the agent doesn't know caching is happening!

In [ ]:
# Create LangChain tools that use your caching implementation
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def search_web(query: str) -> str:
    """
    Search the web for current events, news, and real-time information.
    Results are cached for 30 minutes.

    Args:
        query: The search query

    Returns:
        Search results from the web
    """
    result, is_hit = cached_tool_call(
        "tavily",
        call_tavily,
        ttl_seconds=1800,  # 30 minutes
        query=query
    )
    return result


@tool
def lookup_wikipedia(query: str) -> str:
    """
    Look up encyclopedic information from Wikipedia.
    Use for historical facts, scientific concepts, biographies, and definitions.
    Results are cached for 6 hours.

    Args:
        query: The topic to look up

    Returns:
        Wikipedia article summary
    """
    result, is_hit = cached_tool_call(
        "wikipedia",
        call_wikipedia,
        ttl_seconds=21600,  # 6 hours
        query=query
    )
    return result


# Create the agent with cached tools
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model=llm,
    tools=[search_web, lookup_wikipedia]
)

def ask_agent(question: str) -> str:
    """Helper to query the agent."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

print("Agent created with cached tools!")
print("  - search_web: Web search with 30-min cache")
print("  - lookup_wikipedia: Wikipedia with 6-hour cache")

In [ ]:
# Test the agent - repeated queries will benefit from your cache!
print("Testing Agent with Cached Tools")
print("=" * 60)

# Reset counters
tool_cache.clear()
cache_stats.update({"hits": 0, "misses": 0, "expired": 0})
api_call_counts.update({"tavily": 0, "wikipedia": 0})

# Query 1: Ask about a historical figure (will use Wikipedia)
print("\nQuery 1: Who was Alan Turing?")
print("-" * 40)
response1 = ask_agent("Who was Alan Turing and what is he famous for?")
print(f"Response: {response1[:300]}...")

# Query 2: Ask about current events (will use Tavily)
print("\nQuery 2: Latest AI news")
print("-" * 40)
response2 = ask_agent("What are the latest developments in AI?")
print(f"Response: {response2[:300]}...")

# Query 3: Ask about Alan Turing again (should hit cache!)
print("\nQuery 3: More about Turing (should use cache)")
print("-" * 40)
response3 = ask_agent("Tell me about Alan Turing's contributions to computer science")
print(f"Response: {response3[:300]}...")

# Show cache effectiveness
print("\n" + "=" * 60)
print("AGENT CACHE STATISTICS")
print("=" * 60)
print(f"Cache stats: {cache_stats}")
print(f"API calls: {api_call_counts}")
print(f"\nThe agent automatically benefits from caching!")
print("Repeated or similar queries reuse cached tool results.")